In [1]:
import os
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [3]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "bertopic"
])

0

In [ ]:
import bertopic
import sentence_transformers

print("BERTopic:", bertopic.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("\nBERTopic dependencies loaded successfully.")

BERTopic: 0.17.4
Sentence Transformers: 6.0.1

BERTopic dependencies loaded successfully. ✅


In [7]:
from datasets import load_dataset

# Load Amazon Reviews dataset
dataset = load_dataset("goosmanlei/amazon_reviews_multi", "en")

train_df = dataset["train"].to_pandas()

# Use a reproducible sample for BERTopic
bertopic_df = train_df.sample(
    n=20000,
    random_state=42
).reset_index(drop=True)

# Use cleaned review text
documents = bertopic_df["review_body"].astype(str).tolist()

print("Original training set:", train_df.shape)
print("BERTopic documents:", len(documents))
print("\nSample document:")
print(documents[0])

Original training set: (200000, 8)
BERTopic documents: 20000

Sample document:
Not strong enough to run a small 120v vacuum cleaner, to clean car.


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Embedding model loaded successfully.")
print("Device:", embedding_model.device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8665.40it/s]


Embedding model loaded successfully. ✅
Device: cuda:0


In [9]:
# Generate semantic embeddings for all reviews
embeddings = embedding_model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbeddings generated successfully.")
print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

Batches: 100%|██████████| 313/313 [00:09<00:00, 33.98it/s]



Embeddings generated successfully. ✅
Shape: (20000, 384)
Data type: float32


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words="english",
    min_df=10,
    max_df=0.90,
    ngram_range=(1, 2)
)

# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=100,
    verbose=True
)

print("BERTopic model created successfully. ")

BERTopic model created successfully. ✅


In [12]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    min_df=2,
    ngram_range=(1, 2)
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=100,
    verbose=True
)

print("BERTopic model updated successfully. ✅")

BERTopic model updated successfully. ✅


In [14]:
topics, probs = topic_model.fit_transform(
    documents,
    embeddings
)

print("\nBERTopic training completed. ✅")
print("Number of documents:", len(topics))
print("Number of discovered topics:", len(set(topics)) - (1 if -1 in topics else 0))
print("Number of outliers:", sum(1 for topic in topics if topic == -1))

2026-09-23 08:23:35,963 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-23 08:23:38,781 - BERTopic - Dimensionality - Completed ✓
2026-09-23 08:23:38,784 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-23 08:23:39,948 - BERTopic - Cluster - Completed ✓
2026-09-23 08:23:39,952 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 08:23:40,732 - BERTopic - Representation - Completed ✓



BERTopic training completed. ✅
Number of documents: 20000
Number of discovered topics: 2
Number of outliers: 0


In [15]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=50,
    verbose=True
)

print("BERTopic model updated with min_topic_size=50. ✅")

BERTopic model updated with min_topic_size=50. ✅


In [16]:
topics, probs = topic_model.fit_transform(
    documents,
    embeddings
)

print("\nBERTopic training completed. ✅")
print("Number of documents:", len(topics))
print("Number of discovered topics:", len(set(topics)) - (1 if -1 in topics else 0))
print("Number of outliers:", sum(1 for topic in topics if topic == -1))

2026-09-23 08:25:59,996 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-23 08:26:02,555 - BERTopic - Dimensionality - Completed ✓
2026-09-23 08:26:02,557 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-23 08:26:03,352 - BERTopic - Cluster - Completed ✓
2026-09-23 08:26:03,368 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 08:26:04,101 - BERTopic - Representation - Completed ✓



BERTopic training completed. ✅
Number of documents: 20000
Number of discovered topics: 63
Number of outliers: 7786


In [17]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=50,
    verbose=True
)

print("BERTopic configured with custom HDBSCAN. ✅")

BERTopic configured with custom HDBSCAN. ✅


In [18]:
topics, probs = topic_model.fit_transform(
    documents,
    embeddings
)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for topic in topics if topic == -1)

print("\nBERTopic training completed. ✅")
print("Number of documents:", len(topics))
print("Number of discovered topics:", n_topics)
print("Number of outliers:", n_outliers)
print("Outlier percentage:", round(n_outliers / len(topics) * 100, 2), "%")

2026-09-23 08:27:18,895 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-23 08:27:21,582 - BERTopic - Dimensionality - Completed ✓
2026-09-23 08:27:21,584 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-23 08:27:22,008 - BERTopic - Cluster - Completed ✓
2026-09-23 08:27:22,012 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 08:27:22,578 - BERTopic - Representation - Completed ✓



BERTopic training completed. ✅
Number of documents: 20000
Number of discovered topics: 85
Number of outliers: 6728
Outlier percentage: 33.64 %


In [19]:
import os

model_path = "../../models/topics/bertopic_model"

os.makedirs(model_path, exist_ok=True)

topic_model.save(
    model_path,
    serialization="safetensors",
    save_ctfidf=True
)

print("BERTopic model saved successfully. ✅")
print("Path:", os.path.abspath(model_path))

BERTopic model saved successfully. ✅
Path: d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\topics\bertopic_model


In [20]:
import os
import numpy as np
import joblib

results_path = "../../models/topics/bertopic_results"

os.makedirs(results_path, exist_ok=True)

np.save(
    os.path.join(results_path, "embeddings.npy"),
    embeddings
)

np.save(
    os.path.join(results_path, "topics.npy"),
    np.array(topics)
)

np.save(
    os.path.join(results_path, "probs.npy"),
    np.array(probs)
)

joblib.dump(
    documents,
    os.path.join(results_path, "documents.joblib")
)

print("BERTopic results saved successfully. ✅")
print("Saved to:", os.path.abspath(results_path))

BERTopic results saved successfully. ✅
Saved to: d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\topics\bertopic_results


In [21]:
import os

print("BERTopic model:", os.path.exists("../../models/topics/bertopic_model"))
print("BERTopic results:", os.path.exists("../../models/topics/bertopic_results"))

print("\nSaved result files:")
for file in os.listdir("../../models/topics/bertopic_results"):
    print("✓", file)

BERTopic model: True
BERTopic results: True

Saved result files:
✓ documents.joblib
✓ embeddings.npy
✓ probs.npy
✓ topics.npy


In [2]:
from bertopic import BERTopic

model_path = "../../models/topics/bertopic_model"

topic_model = BERTopic.load(model_path)

print("BERTopic model loaded successfully. ")
print("Number of topics:", len(topic_model.get_topic_info()) - 1)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-23 15:04:21,915 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


BERTopic model loaded successfully. 
Number of topics: 85


In [3]:
topic_info = topic_model.get_topic_info()

print("Number of topics:", len(topic_info) - 1)

display(
    topic_info[
        ["Topic", "Count", "Name"]
    ].head(30)
)

Number of topics: 85


,Topic,Count,Name
0,-1,6728,-1_product_use_like_great
1,0,930,0_book_story_read_characters
2,1,507,1_case_screen_phone_protector
3,2,489,2_sound_ear_headphones_sound quality
4,3,477,3_lights_light_bright_bulbs
5,4,462,4_shoes_shoe_feet_socks
6,5,454,5_quality_price_good_great
7,6,391,6_bag_bags_backpack_strap
8,7,334,7_taste_flavor_tastes_chocolate
9,8,322,8_received_item_delivered_order


In [5]:
import joblib
import os

documents_path = "../../models/topics/bertopic_results/documents.joblib"

documents = joblib.load(documents_path)

print("Documents loaded successfully. ")
print("Number of documents:", len(documents))
print("Sample document:")
print(documents[0])

Documents loaded successfully. 
Number of documents: 20000
Sample document:
Not strong enough to run a small 120v vacuum cleaner, to clean car.


In [6]:
topic_model.reduce_topics(
    documents,
    nr_topics=20
)

topic_info_reduced = topic_model.get_topic_info()

print("Topics after reduction:", len(topic_info_reduced) - 1)

display(
    topic_info_reduced[
        ["Topic", "Count", "Name"]
    ].head(25)
)

2026-09-23 15:07:21,576 - BERTopic - Topic reduction - Reducing number of topics
2026-09-23 15:07:21,731 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 15:07:22,635 - BERTopic - Representation - Completed ✓
2026-09-23 15:07:22,635 - BERTopic - Topic reduction - Reduced number of topics from 86 to 20


Topics after reduction: 19


,Topic,Count,Name
0,-1,6728,-1_product_use_like_great
1,0,1878,0_bag_great_quality_watch
2,1,1626,1_sound_work_charge_battery
3,2,1535,2_size_comfortable_fit_shoes
4,3,1195,3_book_story_read_characters
5,4,995,4_hair_product_skin_smell
6,5,949,5_received_item_box_arrived
7,6,773,6_taste_coffee_flavor_like
8,7,676,7_color_expected_smaller_small
9,8,621,8_case_phone_screen_protector


In [7]:
import os

final_model_path = "../../models/topics/bertopic_final"

os.makedirs(final_model_path, exist_ok=True)

topic_model.save(
    final_model_path,
    serialization="safetensors",
    save_ctfidf=True
)

print("Final BERTopic model saved successfully. ✅")
print("Path:", os.path.abspath(final_model_path))

Final BERTopic model saved successfully. ✅
Path: d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\topics\bertopic_final
